# 📘 Agentic 架构 4：Planning（规划）

本 notebook 探讨 **Planning（规划）** 架构。该模式在 agent 推理中引入**前瞻**：不同于 ReAct 那样逐步响应信息，规划型 agent 先将复杂任务分解为更小、可管理的子目标序列，在采取任何行动**之前**就形成完整「作战计划」。

这种主动方式为多步任务带来结构、可预测性与效率。为突出收益，我们将直接对比**反应式 agent（ReAct）**与新的**规划型 agent**。两者面对同一任务：需收集多条信息后再做最终计算，展示预先制定的计划如何带来更稳健、更直接的解法。

### 定义
**Planning** 架构指 agent 在**开始执行前**就将复杂目标显式分解为详细子任务序列。规划阶段的产出是具体、逐步的计划，随后由 agent 按步骤执行直至得到解。

### 高层工作流

1. **接收目标：** 获得复杂任务。
2. **规划：** 专用「Planner」分析目标，生成有序子任务列表，例如：`["查找事实 A", "查找事实 B", "用 A 和 B 计算 C"]`。
3. **执行：** 「Executor」按顺序执行计划中每一步，按需调用工具。
4. **综合：** 计划全部完成后，最终组件将各步结果合成为连贯的最终答案。

### 适用场景 / 应用
* **多步工作流：** 操作顺序已知且关键的任务，如需拉取数据、处理再汇总的报告生成。
* **项目管理助手：** 将「上线新功能」等大目标拆给不同团队。
* **教学辅导：** 从基础到应用，为学生制定某概念的学习计划。

### 优势与局限
* **优势：**
    * **结构化、可追溯：** 全流程事先展开，过程透明、易调试。
    * **效率：** 对可预测任务可能比 ReAct 更高效，避免步骤间不必要的推理循环。
* **局限：**
    * **对变化脆弱：** 执行中环境突变时，预设计划可能失效；比每步都可改弦更张的 ReAct 适应性差。

## 阶段 0：基础与环境

按惯例安装库，并为 Nebius、LangSmith 与 Tavily 网页搜索配置 API 密钥。

### 步骤 0.1：安装核心库

**本节将做什么：**
安装标准库套件，包括更新的 `langchain-tavily` 包，以消除弃用警告。

In [1]:
# !pip install -q -U langchain-nebius langchain langgraph rich python-dotenv langchain-tavily

### 步骤 0.2：导入库并配置密钥

**本节将做什么：**
导入模块并从 `.env` 加载 API 密钥。

**需要操作：** 在本目录创建 `.env` 并填入密钥：
```
NEBIUS_API_KEY="your_nebius_api_key_here"
LANGCHAIN_API_KEY="your_langsmith_api_key_here"
TAVILY_API_KEY="your_tavily_api_key_here"
```

In [ ]:
import os
import re
from typing import List, Annotated, TypedDict, Optional
from dotenv import load_dotenv

# LangChain components
from langchain_nebius import ChatNebius
from langchain_core.messages import BaseMessage, ToolMessage
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langchain_tavily import TavilySearch

# LangGraph components
from langgraph.graph import StateGraph, END
from langgraph.graph.message import AnyMessage, add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown

# --- API Key and Tracing Setup ---
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Planning (Nebius)"

# Check that the keys are set
for key in ["NEBIUS_API_KEY", "LANGCHAIN_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        print(f"{key} not found. Please create a .env file and set it.")

print("Environment variables loaded and tracing is set up.")

Environment variables loaded and tracing is set up.


## 阶段 1：基线——反应式 agent（ReAct）

要体会规划的价值，先需要基线。我们使用上一 notebook 中构建的 ReAct agent。它很聪明，但**短视**——一次只往前想一步。

### 步骤 1.1：重建 ReAct agent

**本节将做什么：**
快速重建 ReAct agent。其核心是：每次工具调用后，输出仍路由回 agent 自身，使其能根据最新信息重新评估并决定下一步。

In [ ]:
console = Console()

# Define the state for our graphs
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

# 1. Define the base tool from the tavily package
tavily_search_tool = TavilySearch(max_results=2)

# 2. THE FIX: Simplify the custom tool. 
#    The .invoke() method already returns a clean string, so we just pass it through.
@tool
def web_search(query: str) -> str:
    """Performs a web search using Tavily and returns the results as a string."""
    console.print(f"--- TOOL: Searching for '{query}'...")
    results = tavily_search_tool.invoke(query)
    return results

# 3. Define the LLM and bind it to our custom tool
llm = ChatNebius(model="meta-llama/Meta-Llama-3.1-8B-Instruct", temperature=0)
llm_with_tools = llm.bind_tools([web_search])

# 4. Agent node with a system prompt to force one tool call at a time
def react_agent_node(state: AgentState):
    console.print("--- REACTIVE AGENT: Thinking... ---")
    
    messages_with_system_prompt = [
        SystemMessage(content="You are a helpful research assistant. You must call one and only one tool at a time. Do not call multiple tools in a single turn. After receiving the result from a tool, you will decide on the next step.")
    ] + state["messages"]

    response = llm_with_tools.invoke(messages_with_system_prompt)
    
    return {"messages": [response]}

# 5. Use our corrected custom tool in the ToolNode
tool_node = ToolNode([web_search])

# The ReAct graph with its characteristic loop
react_graph_builder = StateGraph(AgentState)
react_graph_builder.add_node("agent", react_agent_node)
react_graph_builder.add_node("tools", tool_node)
react_graph_builder.set_entry_point("agent")
react_graph_builder.add_conditional_edges("agent", tools_condition)
react_graph_builder.add_edge("tools", "agent")

react_agent_app = react_graph_builder.compile()
print("Reactive (ReAct) agent compiled successfully.")

Reactive (ReAct) agent compiled successfully.


### 步骤 1.2：在「偏规划」问题上测试反应式 agent

**本节将做什么：**
给 ReAct agent 一项需要两次独立数据收集再最终计算的任务，检验其在**没有事先计划**的情况下管理多步工作流的能力。

In [4]:
plan_centric_query = """
Find the population of the capital cities of France, Germany, and Italy. 
Then calculate their combined total. 
Finally, compare that combined total to the population of the United States, and say which is larger.
"""

console.print(f"[bold yellow]Testing REACTIVE agent on a plan-centric query:[/bold yellow] '{plan_centric_query}'\n")

final_react_output = None
for chunk in react_agent_app.stream({"messages": [("user", plan_centric_query)]}, stream_mode="values"):
    final_react_output = chunk
    console.print(f"--- [bold purple]Current State Update[/bold purple] ---")
    chunk['messages'][-1].pretty_print()
    console.print("\n")

console.print("\n--- [bold red]Final Output from Reactive Agent[/bold red] ---")
console.print(Markdown(final_react_output['messages'][-1].content))

Testing REACTIVE agent on a plan-centric query: '
Find the population of the capital cities of France, Germany, and Italy. 
Then calculate their combined total. 
Finally, compare that combined total to the population of the United States, and say which is larger.
'

--- Current State Update ---

================================ Human Message =================================


Find the population of the capital cities of France, Germany, and Italy. 
Then calculate their combined total. 
Finally, compare that combined total to the population of the United States, and say which is larger.



--- REACTIVE AGENT: Thinking... ---

--- Current State Update ---

...

--- REACTIVE AGENT: Thinking... ---

--- Current State Update ---

================================== Ai Message ==================================

The population of Paris is approximately 2,165,423. The population of Berlin is approximately 3,580,190. The population of Rome is approximately 4,347,100. The combined total of the population of these three cities is 2,165,423 + 3,580,190 + 4,347,100 = 10,092,713. The population of the United States is approximately 331,449,281. Therefore, the population of the United States is larger than the combined total of the population of Paris, Berlin, and Rome.


--- Final Output from Reactive Agent ---

The population of Paris is approximately 2,165,423. The population of Berlin is approximately 3,580,190. The       
population of Rome is approximately 4,347,100. The combined total of the population of these three cities is       
2,165,423 + 3,580,190 + 4,347,100 = 10,092,713. The population of the United States is approximately 331,449,281.  
Therefore, the population of the United States is larger than the combined total of the population of Paris,       
Berlin, and Rome.

**输出说明：**
ReAct agent 成功完成了任务。从流式输出可追踪其逐步推理：
1. 先决定搜索巴黎人口；
2. 获得结果后纳入上下文，再决定搜索柏林人口；
3. 两条信息齐备后完成计算并给出最终答案。

虽然可行，这种迭代发现并不总是最高效：对这类可预测任务，agent 在步骤间多做了额外的 LLM 推理调用。这为展示规划型 agent 的价值做了铺垫。

## 阶段 2：进阶方案——规划型 agent

接下来构建「先想后做」的 agent：包含专用 **Planner** 生成完整任务列表、**Executor** 执行计划，以及 **Synthesizer** 汇总最终结果。

### 步骤 2.1：定义 Planner、Executor 与 Synthesizer 节点

**本节将做什么：**
创建新 agent 的核心组件：
1. **`Planner`：** 基于 LLM 的节点，接收用户请求并输出结构化计划。
2. **`Executor`：** 接收计划，用工具执行**下一步**并记录结果。
3. **`Synthesizer`：** 最终基于 LLM 的节点，汇总所有收集结果并生成最终答案。

In [5]:
# Pydantic model to ensure the planner's output is a structured list of steps
class Plan(BaseModel):
    """A plan of tool calls to execute to answer the user's query."""
    steps: List[str] = Field(description="A list of tool calls that, when executed, will answer the query.")

# Define the state for the planning agent
class PlanningState(TypedDict):
    user_request: str
    plan: Optional[List[str]]
    intermediate_steps: List[ToolMessage]
    final_answer: Optional[str]

def planner_node(state: PlanningState):
    """Generates a plan of action to answer the user's request."""
    console.print("--- PLANNER: Decomposing task... ---")
    planner_llm = llm.with_structured_output(Plan)
    
    # THE FIX: A much more explicit prompt with a clear example (few-shot prompting)
    prompt = f"""You are an expert planner. Your job is to create a step-by-step plan to answer the user's request.
        Each step in the plan must be a single call to the `web_search` tool.

        **Instructions:**
        1. Analyze the user's request.
        2. Break it down into a sequence of simple, logical search queries.
        3. Format the output as a list of strings, where each string is a single valid tool call.

        **Example:**
        Request: "What is the capital of France and what is its population?"
        Correct Plan Output:
        [
            "web_search('capital of France')",
            "web_search('population of Paris')"
        ]

        **User's Request:**
        {state['user_request']}
    """

    plan_result = planner_llm.invoke(prompt)
    # Use plan_result.steps, not plan.steps to avoid confusion with the variable name 'plan'
    console.print(f"--- PLANNER: Generated Plan: {plan_result.steps} ---")
    return {"plan": plan_result.steps}

def executor_node(state: PlanningState):
    """Executes the next step in the plan."""
    console.print("--- EXECUTOR: Running next step... ---")
    plan = state["plan"]
    next_step = plan[0]
    
    # Robust regex to handle both single and double quotes
    match = re.search(r"(\w+)\((?:\"|\')(.*?)(?:\"|\')\)", next_step)
    if not match:
        tool_name = "web_search"
        query = next_step
    else:
        tool_name, query = match.groups()[0], match.groups()[1]
    
    console.print(f"--- EXECUTOR: Calling tool '{tool_name}' with query '{query}' ---")
    
    result = tavily_search_tool.invoke(query)
    
    # We still create a ToolMessage, but the tool call itself is now safe.
    tool_message = ToolMessage(
    content=str(result),
    name=tool_name,
    tool_call_id=f"manual-{hash(query)}"
    )
    
    return {
        "plan": plan[1:], # Pop the executed step from the plan
        "intermediate_steps": state["intermediate_steps"] + [tool_message]
    }

def synthesizer_node(state: PlanningState):
    """Synthesizes the final answer from the intermediate steps."""
    console.print("--- SYNTHESIZER: Generating final answer... ---")
    
    context = "\n".join([f"Tool {msg.name} returned: {msg.content}" for msg in state["intermediate_steps"]])
    
    prompt = f"""You are an expert synthesizer. Based on the user's request and the collected data, provide a comprehensive final answer.
    
    Request: {state['user_request']}
    Collected Data:
    {context}
    """
    final_answer = llm.invoke(prompt).content
    return {"final_answer": final_answer}

print("Planner, Executor, and Synthesizer nodes defined.")

Planner, Executor, and Synthesizer nodes defined.


### 步骤 2.2：构建规划型 agent 图

**本节将做什么：**
将新节点组装成图。流程为：`Planner` -> `Executor`（循环）-> `Synthesizer`。

In [6]:
def planning_router(state: PlanningState):
    if not state["plan"]:
        console.print("--- ROUTER: Plan complete. Moving to synthesizer. ---")
        return "synthesize"
    else:
        console.print("--- ROUTER: Plan has more steps. Continuing execution. ---")
        return "execute"

planning_graph_builder = StateGraph(PlanningState)
planning_graph_builder.add_node("plan", planner_node)
planning_graph_builder.add_node("execute", executor_node)
planning_graph_builder.add_node("synthesize", synthesizer_node)

planning_graph_builder.set_entry_point("plan")
planning_graph_builder.add_conditional_edges("plan", planning_router, {"execute": "execute", "synthesize": "synthesize"}) # Route after planning
planning_graph_builder.add_conditional_edges("execute", planning_router, {"execute": "execute", "synthesize": "synthesize"})
planning_graph_builder.add_edge("synthesize", END)

planning_agent_app = planning_graph_builder.compile()
print("Planning agent compiled successfully.")

Planning agent compiled successfully.


## 阶段 3：对比实验

在同一任务上运行新的规划型 agent，将其执行流程与最终输出与反应式 agent 对比。

In [7]:
console.print(f"[bold green]Testing PLANNING agent on the same plan-centric query:[/bold green] '{plan_centric_query}'\n")

# Remember to initialize the state correctly, especially the list for intermediate steps
initial_planning_input = {"user_request": plan_centric_query, "intermediate_steps": []}

final_planning_output = planning_agent_app.invoke(initial_planning_input)

console.print("\n--- [bold green]Final Output from Planning Agent[/bold green] ---")
console.print(Markdown(final_planning_output['final_answer']))

Testing PLANNING agent on the same plan-centric query: '
Find the population of the capital cities of France, Germany, and Italy. 
Then calculate their combined total. 
Finally, compare that combined total to the population of the United States, and say which is larger.
'

--- PLANNER: Decomposing task... ---

--- PLANNER: Generated Plan: ["web_search('population of the capital of France')", "web_search('population of the 
capital of Germany')", "web_search('population of the capital of Italy')", "web_search('population of the United 
States')", "web_search('add  + population of France + population of Germany + population of Italy')", 
"web_search('compare + combined total + population of the United States')"] ---

--- ROUTER: Plan has more steps. Continuing execution. ---

--- EXECUTOR: Running next step... ---

...

--- EXECUTOR: Calling tool 'web_search' with query 'compare + combined total + population of the United States' ---

--- ROUTER: Plan complete. Moving to synthesizer. ---

--- SYNTHESIZER: Generating final answer... ---

--- Final Output from Planning Agent ---

Based on the collected data, here are the final answers to the user's request:                                     

 1 The population of the capital cities of France, Germany, and Italy are:                                         
    • Paris, France: 2,048,472 (according to Instagram)                                                            
    • Berlin, Germany: 3,800,000 (according to Google Arts & Culture)                                              
    • Rome, Italy: 2,800,000 (according to Instagram)                                                              
 2 The combined total of the population of these three capital cities is:                                          
    • 2,048,472 (Paris) + 3,800,000 (Berlin) + 2,800,000 (Rome) = 8,648,472                                        
 3 The population of the United States is approximately 343,603,404 (according to Macrotrends).                    
 4 The combined total of the population of the three capital cities (8,648,472) is significantly smaller than the  
   population of the United States (343,603,404).                                                                  

Therefore, the population of the United States is larger than the combined total of the population of the capital  
cities of France, Germany, and Italy.

**输出说明：**
过程差异一目了然。第一步就是 `Planner` 生成完整、显式的计划，例如：`['web_search("population of Paris")', 'web_search("population of Berlin")']`。

随后 agent 按该计划逐步执行，无需在步骤间停下来「再想」。这一过程：
- **更透明：** 甚至在开始前就能看到完整策略。
- **更稳健：** 不太容易跑偏，因为遵循明确指令。
- **可能更高效：** 减少步骤间用于推理的额外 LLM 调用。

这展示了在可事先确定步骤的任务上，规划能力的威力。

## 阶段 4：量化评估

为形式化对比，使用 LLM-as-a-Judge 对两个 agent 打分，侧重解题过程的质量与效率。

In [8]:
class ProcessEvaluation(BaseModel):
    """Schema for evaluating an agent's problem-solving process."""
    task_completion_score: int = Field(description="Score 1-10 on whether the agent successfully completed the task.")
    process_efficiency_score: int = Field(description="Score 1-10 on the efficiency and directness of the agent's process. A higher score means a more logical and less roundabout path.")
    justification: str = Field(description="A brief justification for the scores.")

judge_llm = llm.with_structured_output(ProcessEvaluation)

def evaluate_agent_process(query: str, final_state: dict):
    # For the ReAct agent, the trace is in 'messages'. For Planning, it's in 'intermediate_steps'.
    if 'messages' in final_state:
        trace = "\n".join([f"{m.type}: {str(m.content)}" for m in final_state['messages']])
    else:
        trace = f"Plan: {final_state.get('plan', [])}\nSteps: {final_state.get('intermediate_steps', [])}"
        
    prompt = f"""You are an expert judge of AI agents. Evaluate the agent's process for solving the task on a scale of 1-10.
    Focus on whether the process was logical and efficient.
    
    **User's Task:** {query}
    **Full Agent Trace:**\n```\n{trace}\n```
    """
    return judge_llm.invoke(prompt)

console.print("--- Evaluating Reactive Agent's Process ---")
react_agent_evaluation = evaluate_agent_process(plan_centric_query, final_react_output)
console.print(react_agent_evaluation.model_dump())

console.print("\n--- Evaluating Planning Agent's Process ---")
planning_agent_evaluation = evaluate_agent_process(plan_centric_query, final_planning_output)
console.print(planning_agent_evaluation.model_dump())

--- Evaluating Reactive Agent's Process ---

{
    'task_completion_score': 10,
    'process_efficiency_score': 8,
    'justification': 'The agent successfully completed the task by finding the population of the capital cities of 
France, Germany, and Italy, and then calculating their combined total. However, the process was not entirely 
efficient as the agent made multiple tool calls to find the population of each city, and the answers were not used 
effectively to calculate the combined total. The agent could have used the information from the first tool call to 
calculate the combined total and then compared it to the population of the United States. Additionally, the agent 
could have used more efficient tools or methods to find the population of the cities, such as using a single tool 
that provides the population of multiple cities at once.'
}

--- Evaluating Planning Agent's Process ---

{
    'task_completion_score': 8,
    'process_efficiency_score': 6,
    'justification': 'The agent was able to find the population of the capital cities of France, Germany, and 
Italy, and calculate their combined total. However, the process was not entirely efficient as the agent had to 
perform multiple web searches to find the correct information. The agent also did not directly compare the combined
total to the population of the United States, but rather had to perform another web search to find the relevant 
information. Overall, the agent was able to complete the task, but the process could have been more efficient.'
}

**输出说明：**
评判分数量化了两种方式的差异。两者最终都可能得到较高的 `task_completion_score`。但**规划型 agent** 的 `process_efficiency_score` 会明显更高；评判理由会强调：相对 ReAct 逐步探索，事先制定的计划是更直接、更符合逻辑的路径。

该评估验证了我们的假设：在解路径可预测的问题上，Planning 架构提供更结构化、透明且高效的方式。

## 小结

本 notebook 实现了 **Planning** 架构，并与 **ReAct** 直接对照。强制 agent 在执行前先构造完整计划，在定义明确的多步任务上可显著提升透明度、稳健性与效率。

ReAct 擅长探索性、下一步未知的场景；Planning 则适合能预先绘出路径的情形。理解这一权衡对系统设计至关重要：为合适的问题选对架构，是构建有效智能 agent 的关键技能。Planning 模式是工具箱中不可或缺的一环，为复杂、可预测的工作流提供所需结构。